In [0]:
%run ../00_setup

In [0]:

#Obtenemos el diccionario de configuración del endpoint que se va a almacenar
def get_raw_config(entity_name: str) -> dict:
    df = spark.sql(f"""
        SELECT * FROM {CATALOGO}.{ESQUEMA_CONTROL}.raw_ingestion_config
        WHERE entity_name = '{entity_name}' AND is_active = true
    """)
    filas = df.collect()
    if len(filas) == 0:
        raise ValueError(f"No existe configuración activa para '{entity_name}'")
    return filas[0].asDict()

#Obtener un diccionario anidado clave-valor ({param_name: param_value}) con todos los parámetros que necesita la API que se va a ingestar. También se incluye el tipo de parámetro de offset y el de page_size  nombrado por la API correspondiente.
def get_api_parameters(entity_name: str) -> dict:
    df = spark.sql(f"""
        SELECT param_name, param_value, param_type
        FROM {CATALOGO}.{ESQUEMA_CONTROL}.raw_api_parameters
        WHERE entity_name = '{entity_name}'
    """)
    filas = df.collect()
    params_dict = {f.param_name: f.param_value for f in filas}
    offset_param = next((f.param_name for f in filas if f.param_type == "offset"), None)
    page_size_param = next((f.param_name for f in filas if f.param_type == "page_size"), None)
    return {"params": params_dict, "offset_param_name": offset_param, "page_size_param_name": page_size_param}